# visualizar_aleatorio_reduzida — Google Colab

Escolhe **aleatoriamente** uma imagem reduzida gerada por
`compilar_reduzir_classificadas_colab.ipynb` e mostra o **original reduzido (RGB)**
ao lado do **predict reduzido**.

Reexecute a última célula quantas vezes quiser para sortear outra imagem.

---
**Drive montado em `/content/drive`.**

In [ ]:
# ── Célula 1: Montar Drive e instalar dependências ───────────────────────────
from google.colab import drive
drive.mount('/content/drive')

!pip install -q rasterio matplotlib

In [ ]:
# ── Célula 2: PARÂMETROS ─────────────────────────────────────────────────────
REDUCED_DIR    = "/content/drive/MyDrive/DL_fotovoltaica/tif_reduzidas_2025"  # saída do notebook de redução
RGB_BANDS      = (3, 2, 1)   # bandas 1-based p/ composição RGB
THRESHOLD_PRED = 0.5         # usado só se o predict for float (senão, ==1)
SEED           = None        # None = sorteio diferente a cada execução; int = reprodutível

print("Parâmetros carregados.")
print(f"  REDUCED_DIR: {REDUCED_DIR}")

In [ ]:
# ── Célula 3: Imports e funções ──────────────────────────────────────────────
import random
from pathlib import Path
import numpy as np
import rasterio
import matplotlib.pyplot as plt

reduced_dir = Path(REDUCED_DIR)

def ler_rgb(tif_path: Path, rgb_bands=RGB_BANDS) -> np.ndarray:
    with rasterio.open(tif_path) as src:
        bands = [src.read(b) for b in rgb_bands]
    rgb = np.stack(bands, axis=-1).astype(np.float32)
    for i in range(rgb.shape[-1]):
        ch = rgb[..., i]
        lo, hi = np.percentile(ch, (2, 98))
        rgb[..., i] = np.clip((ch - lo) / (hi - lo + 1e-6), 0, 1)
    return rgb

def ler_mascara(pred_path: Path) -> np.ndarray:
    with rasterio.open(pred_path) as src:
        data = src.read(1)
    return (data == 1) if data.dtype == np.uint8 else (data > THRESHOLD_PRED)

def listar_reduzidas(reduced_dir: Path):
    """Pares (img_reduzido, pred_reduzido) existentes em reduced_dir."""
    pares = []
    for img in sorted(reduced_dir.glob('*_img_reduzido.tif')):
        stem = img.stem[:-len('_img_reduzido')]
        pred = reduced_dir / f'{stem}_pred_reduzido.tif'
        if pred.exists():
            pares.append((stem, img, pred))
    return pares

if SEED is not None:
    random.seed(SEED)

pares = listar_reduzidas(reduced_dir)
print(f'{len(pares)} imagem(ns) reduzida(s) disponível(is) em {reduced_dir}')

In [ ]:
# ── Célula 4: Sortear e visualizar (reexecute p/ sortear outra) ──────────────
def mostrar_reduzida(stem: str, img_path: Path, pred_path: Path):
    rgb   = ler_rgb(img_path)
    mask  = ler_mascara(pred_path)
    n_pos = int(mask.sum())
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    ax[0].imshow(rgb)
    ax[0].set_title(f'Original reduzido\n{img_path.name}  {rgb.shape[1]}x{rgb.shape[0]}px')
    ax[0].axis('off')
    ax[1].imshow(mask, cmap='gray')
    ax[1].set_title(f'Predict reduzido\n{pred_path.name}  ({n_pos:,} px = 1)')
    ax[1].axis('off')
    plt.suptitle(f'{stem}', fontsize=13)
    plt.tight_layout()
    plt.show()

if not pares:
    print('Nenhuma imagem reduzida encontrada. Rode o notebook de redução primeiro.')
else:
    mostrar_reduzida(*random.choice(pares))

In [ ]:
# ── Célula 5: Escolher por nome específico ───────────────────────────────────
# Informe o identificador (stem) da imagem, com ou sem o sufixo. Ex.:
#   "00000000000000000005_2023"  |  "..._img_reduzido"  |  "..._img_reduzido.tif"
NOME_ESCOLHIDO = "00000000000000000005_2023"

def _normaliza_stem(nome: str) -> str:
    nome = Path(nome).name
    for suf in ('_img_reduzido.tif', '_pred_reduzido.tif', '_img_reduzido', '_pred_reduzido', '.tif'):
        if nome.endswith(suf):
            nome = nome[:-len(suf)]
            break
    return nome

alvo_stem = _normaliza_stem(NOME_ESCOLHIDO)
match = next((p for p in pares if p[0] == alvo_stem), None)

if match is None:
    print(f'Nenhuma imagem reduzida com stem "{alvo_stem}".')
    disp = [p[0] for p in pares]
    print(f'{len(disp)} disponível(is). Exemplos:')
    for s in disp[:20]:
        print('  ', s)
else:
    mostrar_reduzida(*match)